In [28]:
import os
import sys
from pathlib import Path
from collections import Counter

from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings



In [ ]:
# Check project root
print("CWD:", os.getcwd())
print("First sys.path entry:", sys.path[0])
print("Is project root on sys.path?", Path.cwd().parent in map(Path, sys.path))

In [29]:
# Set project root
PROJECT_ROOT = Path.cwd().parent  # adjust as needed

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("CWD:", os.getcwd())
print("First sys.path entry:", sys.path[0])
print("Is project root on sys.path?", Path.cwd().parent in map(Path, sys.path))

CWD: /Users/alexis/Desktop/Learning/Projects/202502_Custom_Chatbot/notebooks
First sys.path entry: /Users/alexis/Desktop/Learning/Projects/202502_Custom_Chatbot
Is project root on sys.path? True


### 1. Check ChromaDB build

In [30]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from collections import Counter
from pathlib import Path

# === CONFIG ===
PERSIST_DIR = str(Path("../chroma_db/cc_faq_openai"))
COLLECTION_NAME = "cc_faq"
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

# === Load Vector Store ===
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key = OPENAI_API_KEY
)

vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    persist_directory=PERSIST_DIR,
    embedding_function=embeddings
)

print("Vector DB loaded.\n")


Vector DB loaded.



In [31]:
# 1️. Check metadata structure & segment distribution
raw = vectorstore._collection.get(include=["metadatas"], limit=2000)
metadatas = raw["metadatas"]

segments = [m.get("segment", "MISSING") for m in metadatas]
segment_counts = Counter(segments)

print("Segment distribution:")
print(segment_counts)
print()

if "MISSING" in segment_counts:
    print("WARNING: Some chunks missing 'segment' metadata.\n")
else:
    print("All chunks contain 'segment' metadata.\n")


Segment distribution:
Counter({'generic': 598, 'personal': 457, 'business': 191, 'corporate': 24})

All chunks contain 'segment' metadata.



In [ ]:
# 2. Test filtered retrieval
def test_query(query, segment):
    print(f"\n=== Query: '{query}' | Segment: {segment} ===")

    filter_condition = {
        "$or": [
            {"segment": segment},
            {"segment": "generic"}
        ]
    }

    results = vectorstore.similarity_search_with_relevance_scores(
        query,
        k=10,
        filter=filter_condition
    )

    for i, (doc, score) in enumerate(results, 1):
        doc_segment = doc.metadata.get('segment')
        doc_url = doc.metadata.get('url')
        # doc_content = doc.page_content[:2000]
        print(f"{i}. Score: {score:.4f} | Segment: {doc_segment} | URL: {doc_url}")

# Try a few probes
# test_query("Earn qantas points on credit cards", "personal")
test_query("What is the interest rate on business low rate card?", "business")


In [ ]:
results = vectorstore.similarity_search_with_relevance_scores(
    "Cash Advances ATM fee cash advance interest rarte rates and fee",
    k=5,
    filter={"url":"https://www.commbank.com.au/business/business-credit-cards/business-low-rate-credit-card.html"}
)

for i, (doc, score) in enumerate(results, 1):
    print(f"{i}. Score: {score:.4f} | Chunk Preview: \n{doc.page_content[:200]}\n")

In [ ]:
results = vectorstore.similarity_search_with_relevance_scores(
    "Earn qantas points on credit cards",
    k=5,
    filter={"$or": [{"segment": "personal"}, {"segment": "generic"}]}
)

for i, (doc, score) in enumerate(results, 1):
    text = doc.page_content.lower()
    print(f"\n{i}. score={score:.4f} segment={doc.metadata.get('segment')} url={doc.metadata.get('url')}")
    print("contains 'qantas'?", "qantas" in text)
    print("contains 'points'?", "points" in text)
    print("snippet:", doc.page_content[:300].replace("\n", " "))

In [ ]:
target_urls = [
    "https://www.commbank.com.au/credit-cards/qantas-frequent-flyer.html",
    "https://www.commbank.com.au/credit-cards/smart-credit-card-qantas.html",
    "https://www.commbank.com.au/credit-cards/ultimate-credit-card-qantas.html",
    "https://www.commbank.com.au/credit-cards/awards/earn-points.html",
    "https://www.commbank.com.au/business/business-credit-cards/business-awards-credit-card.html",
    "https://www.commbank.com.au/business/business-credit-cards/business-awards-platinum-credit-card.html",
]

col = vectorstore._collection
for url in target_urls:
    res = col.get(where={"url": url}, include=["metadatas"], limit=5)
    print(url, "-> chunks found:", len(res["ids"]))

### 2. Test LLM Chain

In [ ]:
# Test LLMChain
from langchain_rag.rag_pipeline import get_vectorstore, llm_chain, REFUSAL_TEXT

vs = get_vectorstore()
q = "What is the interest rate on a Low Rate credit card?"

# 1) Retrieve (with scores if you want)
docs_and_scores = vs.similarity_search_with_relevance_scores(q, k=5)
docs = [d for d, _ in docs_and_scores]

print("retrieved docs", len(docs_and_scores))

# 2) Build context
context = "\n\n---\n\n".join(d.page_content for d in docs)
print("built context",len(context))

# 3) Call LLMChain with correct keys
resp = llm_chain.invoke({"context": context, "question": q})
answer = resp.content

print("Answer:\n", answer)

# Optional: show sources
for d, score in docs_and_scores[:3]:
    print("\n", score, d.metadata.get("title"), d.metadata.get("url"))